## Первичный анализ

В этом проекте построена **линейная регрессия**, проведено её сравнение с **Lasso-регрессией**

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
#Считываем датафрейм 
data = pd.read_csv('diamonds.csv')
data.head(5)

,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [4]:
#Проверяем на наличие пустых значений (их нет)
data.isna().sum()

Unnamed: 0    0
carat         0
cut           0
color         0
clarity       0
depth         0
table         0
price         0
x             0
y             0
z             0
dtype: int64

In [6]:
#Удаляем неинформативный столбец
data.drop('Unnamed: 0', axis=1, inplace=True)

Столбец **Unnamed: 0** было необходимо удалить, потому что по сути он повторяет по функционалу индексы питона

## Построение модели

Допустим нам необходимо построить модель, которая будет предсказывать стоимость бриллианта, основываясь на **предикторах**  
Для этого посмотрим корреляцию между признаками

In [7]:
#Целиком корреляционная матрица
data.corr(numeric_only=True)

,carat,depth,table,price,x,y,z
carat,1.000000,0.028224,0.181618,0.921591,0.975094,0.951722,0.953387
depth,0.028224,1.000000,-0.295779,-0.010647,-0.025289,-0.029341,0.094924
table,0.181618,-0.295779,1.000000,0.127134,0.195344,0.183760,0.150929
price,0.921591,-0.010647,0.127134,1.000000,0.884435,0.865421,0.861249
x,0.975094,-0.025289,0.195344,0.884435,1.000000,0.974701,0.970772
y,0.951722,-0.029341,0.183760,0.865421,0.974701,1.000000,0.952006
z,0.953387,0.094924,0.150929,0.861249,0.970772,0.952006,1.000000


Можно заметить, что у многих признаков есть высокая корреляция с целевой переменной **(Target)**. Однако, есть нюанс, между всеми этими признаками:**carat, depth, x, y, z** есть высокая корреляция.  
Это вызывает явление мультиколлинеарности, что логично: размеры **(x, y, z)** очевидно связаны с весом бриллианта **(carat)**.  
В связи с этой проблемой и возникает необходимость построить **Lasso-регрессию**. Но для начала посмотрим её качество на **обычной линейной**

In [9]:
data.corr(numeric_only=True)['price'].sort_values(key=abs)
#отсортируем по модулю, чтоб выбрать наибольшие корр. по модулю

depth   -0.010647
table    0.127134
z        0.861249
y        0.865421
x        0.884435
carat    0.921591
price    1.000000
Name: price, dtype: float64

In [10]:
data.dtypes

carat      float64
cut            str
color          str
clarity        str
depth      float64
table      float64
price        int64
x          float64
y          float64
z          float64
dtype: object

Перед построением линейной регрессии необходимо закодировать категориальные признаки. По правилам, категориальные порядковые - нужно закодировать через **LabelEncoder**(он сохраняет эффект порядка, а не создаёт бинарные признаки). Например **cut = ideal** лучше, чем **good**  
Аналогично с **clarity**.  

**Color** же в свою очередь - определённо категориально номинальная переменная. Её закодируем через **One-hot encoding**

In [13]:
#Закодировали категориальные порядковые 
from sklearn.preprocessing import LabelEncoder


data['clarity_encoded'] = LabelEncoder().fit_transform(data['clarity'])
data['cut_encoded'] = LabelEncoder().fit_transform(data['cut'])

In [14]:
#Закодировали категориальные номинальные
df_clean = pd.get_dummies(data, columns=['color'], drop_first=True, dtype='int')

In [16]:
#А теперь уберём те признаки, на основе которыех был применён LabelEncode
df_clean.drop('cut', axis=1, inplace=True)
df_clean.drop('clarity', axis=1, inplace=True)

А теперь построим **линейную регрессию**

In [21]:
from sklearn.model_selection import train_test_split


X = df_clean.drop('price', axis=1)
y = df_clean['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [23]:
#Масштабируем вещественные признаки, чтобы все вещественные числа были в одном масштабе
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
nums = ['carat', 'depth', 'table', 'x', 'y', 'z']
X_tr_sc = X_train.copy()
X_te_sc = X_test.copy() # на всякий пожарный сделаем через копии

X_tr_sc[nums] = sc.fit_transform(X_train[nums])
X_te_sc[nums] = sc.fit_transform(X_test[nums])

Теперь оценим качество модели через **MSE** (чтобы потом сравнить его с Lasso-регрессией)

In [24]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


lr = LinearRegression()
model = lr.fit(X_tr_sc, y_train)
y_tr_pred = lr.predict(X_tr_sc)
y_te_pred = lr.predict(X_te_sc)

print(f'MSE на тренировочной выборке {mean_squared_error(y_train, y_tr_pred)}')
print(f'MSE на тестовой выборке {mean_squared_error(y_test, y_te_pred)}')

MSE на тренировочной выборке 1765060.730258238
MSE на тестовой выборке 1760149.3314122262


## Lasso

Для построения Lasso-регрессии нужно знать оптимальны **a**(коэффициент регуляризации). К сожалению, это гиперпараметр, который надо находить эмпирически.  
Чем и займёмся

In [28]:
from sklearn.linear_model import LassoCV #нужен для кросс-валидации


alphs = [a for a in range(1, 101)]
lc = LassoCV(alphas=alphs, cv=5)
lc.fit(X_tr_sc, y_train)

matrix = lc.mse_path_
avg = np.mean(matrix, axis=1)
a = lc.alphas_
idx = np.argmin(avg) #наибольшее качество у того значения, при котором ошибка минимальная

print(f'Лучшее значение альфа: {a[idx]}')

Лучшее значение альфа: 1


In [30]:
#Обучаем итоговую Lasso-регрессию
from sklearn.linear_model import Lasso


ls = Lasso(1).fit(X_tr_sc, y_train)
coef_df = pd.DataFrame({'predictor': X_tr_sc.columns, 'coefficient': ls.coef_})
sort_coef_df = coef_df.sort_values('coefficient', key=abs)
sort_coef_df

,predictor,coefficient
5,z,0.000000
4,y,0.000000
7,cut_encoded,74.025949
9,color_F,-90.696933
8,color_E,-120.708682
10,color_G,-177.264776
2,table,-200.971387
1,depth,-217.598288
6,clarity_encoded,278.732592
11,color_H,-745.965769


In [31]:
#Оценим качество регрессии по MSE
y_pred_l = ls.predict(X_te_sc)
print(f'MSE Lasso-регрессии: {round(mean_squared_error(y_test, y_pred_l), 2)}')

MSE Lasso-регрессии: 1760407.33
